In [9]:
import { ChatGoogle } from "npm:@langchain/google";

import { GoogleGenerativeAIEmbeddings } from "npm:@langchain/google-genai";

import { MemoryVectorStore } from "npm:@langchain/classic/vectorstores/memory";

import { Document } from "npm:@langchain/core/documents";

import { parse } from "jsr:@std/dotenv";


In [10]:
const env = parse(await Deno.readTextFile(".env"));

const GOOGLE_API_KEY = env.GOOGLE_API_KEY;
const API_URL = env.RESTAURANT_API_URL;


In [11]:
const model = new ChatGoogle({
  model: "gemini-3.1-flash-lite",
  apiKey: GOOGLE_API_KEY,
  temperature: 0,
});


In [12]:
type RestaurantCategory =
  | "한식"
  | "일식"
  | "중식"
  | "양식"
  | "세계요리"
  | "특별한 술집"
  | "전통차/커피전문점"
  | "디저트/베이커리";

type Restaurant = {
  OPENDATA_ID: string;
  BZ_NM: string;
  GNG_CS: string;
  FD_CS: string;
  TLNO: string;
  MBZ_HR: string;
  SEAT_CNT: string;
  PKPL: string;
  HP: string;
  PSB_FRN: string;
  BKN_YN: string;
  INFN_FCL: string;
  BRFT_YN: string;
  DSSRT_YN: string;
  MNU: string;
  SMPL_DESC: string;
  SBW: string;
  BUS: string;
};

// 외부api 호출 함수
async function fetchRestaurants(district: string): Promise<Restaurant[]> {
  const url = new URL(API_URL);

  url.searchParams.set("addr", district);

  const response = await fetch(url);

  if (!response.ok) {
    throw new Error(`API 요청 실패: ${response.status}`);
  }

  const data = await response.json();

  // 실제 API 응답 구조에 맞게 수정
  return data.data as Restaurant[];
}

// 카테고리 필터 함수
function filterByCategory(
  restaurants: Restaurant[],
  category: RestaurantCategory | null,
): Restaurant[] {
  if (category === null) {
    return restaurants;
  }

  return restaurants.filter((restaurant) => restaurant.FD_CS === category);
}


* 임베딩
문장의 의미를 숫자 배열로 바꾸는 것
숫자배열->벡터
숫자배열로 바꾸는 이유 : 수치를 통해 문장의 유사도를 비교할 수 있다
문장이 정확히 같은 단어를 사용하지 않더라도 의미가 비슷한지 비교하기 위해서

In [13]:
const embeddings = new GoogleGenerativeAIEmbeddings({
  model: "gemini-embedding-001",
  apiKey: GOOGLE_API_KEY,
});


레스토랑데이터를 문서형태로 변환

In [14]:
function createRestaurantDocument(restaurant: Restaurant) {
  const content = `
${restaurant.BZ_NM}은(는)
${restaurant.GNG_CS}에 위치한
${restaurant.FD_CS} 음식점입니다.

${restaurant.SMPL_DESC}

좌석 정보는 ${restaurant.SEAT_CNT}입니다.
주차 정보는 ${restaurant.PKPL}입니다.
예약은 ${restaurant.BKN_YN}합니다.

${restaurant.SBW}
${restaurant.BUS}
  `.trim();

  return new Document({
    pageContent: content,

    metadata: {
      id: restaurant.OPENDATA_ID,

      name: restaurant.BZ_NM,

      category: restaurant.FD_CS,

      address: restaurant.GNG_CS,
    },
  });
}


In [39]:
const restaurants = await fetchRestaurants("중구");
const categoryFiltered = filterByCategory(restaurants, "일식");

const documents = categoryFiltered.map(createRestaurantDocument);

console.log(documents[0]);

Document {
  pageContent: "톤톤 돈카츠은(는)\n" +
    "대구광역시 중구 대봉동 4-1에 위치한\n" +
    "일식 음식점입니다.\n" +
    "\n" +
    "‘톤톤 돈가츠’는 작지만 정성 가득한 수제 돈카츠 전문점입니다.\n" +
    "\n" +
    "좌석 정보는 20석입니다.\n" +
    "주차 정보는 김광석길공영주차장이용(유료)입니다.\n" +
    "예약은 가능합니다.\n" +
    "\n" +
    "지하철 2호선 경대병원역 3번 출구에서 도보로 약 483m 거리.\n" +
    "버스 정류장은 방천시장(김광석길)앞 정류장이 가장 가깝습니다.",
  metadata: {
    id: "1874",
    name: "톤톤 돈카츠",
    category: "일식",
    address: "대구광역시 중구 대봉동 4-1"
  },
  id: undefined
}


청킹 : 문서가 너무 길어서 한 벡터에 여러 의미가 섞일 때 나누는 기술
내용이 짧을 경우 오히려 정확도가 떨어질수있다



벡터DB생성

① Indexing
데이터를 검색 가능하게 준비

In [40]:
const vectorStore = await MemoryVectorStore.fromDocuments(
  documents,
  embeddings,
);


② Retrieval
사용자 질문과 가까운 문서 검색
"부모님 모시고 갈 조용한 식당"을 의미적으로 가까운 문서를 찾아낸다.

In [41]:
const query = "같은 친구들과 같이 갈수있는곳";

const results = await vectorStore.similaritySearch(query, 3);

In [ ]:
for (const result of results) {
  console.log("식당:", result.metadata.name);

  console.log(result.pageContent);

  console.log("-------------------");
}


식당: 힛또
힛또은(는)
대구광역시 중구 대봉동 214, 11층에 위치한
일식 음식점입니다.

대백프라자 10층에 위치하고 있는 힛또는 스위스까스, 나가사끼탕면(생면)등 인기 있는 메뉴를 가지고 있으며 업주가 계속적인 연구 노력하는 업체이다.<br />

좌석 정보는 116입니다.
주차 정보는 대백프라자주차장입니다.
예약은 가능합니다.

지하철 3호선 대봉교역 4번 출구 약 110m
버스 정류장은 대백프라자건너 정류장이 가장 가깝습니다.
-------------------
식당: 삼삼구이초밥
삼삼구이초밥은(는)
대구광역시 중구 남산동 921-5에 위치한
일식 음식점입니다.

활어회, 장어구이에 회초밥과 우동까지 저렴한 가격에 한 끼 식사를 겸한 술 한 잔 하기 딱 좋은 집이다. 곁들임 안주의 종류와 양이 푸짐하기로 유명하다.

좌석 정보는 70석입니다.
주차 정보는 유료주차(1시간무료)입니다.
예약은 가능합니다.

지하철 1호선 반월당역 3번 출구 약 75m
버스 정류장은 반월당역(1번출구)1 정류장이 가장 가깝습니다.
-------------------
식당: 남산에
남산에은(는)
대구광역시 중구 남산동 3006, 효성해링턴 상가 1층 304동 105-2호에 위치한
일식 음식점입니다.

남산에는 지리산 흑돼지를 사용한 두툼한 돈까스로 유명한 돈까스 전문점입니다.

좌석 정보는 13석입니다.
주차 정보는 탑마트주차장이용가능(1시간무료)입니다.
예약은 불가능합니다.

지하철 1,2호선 반월당역 2번 출구에서 도보로 약 391m 거리.
버스 정류장은 반월당역(2번출구) 정류장이 가장 가깝습니다.


-------------------


Retriever로 사용

In [43]:
const retriever = vectorStore.asRetriever(3);
const results = await retriever.invoke("부모님과 조용하게 식사할 곳");


In [44]:
console.log(results.map((document) => document.metadata.name));


[ "사야까", "종로초밥", "디귿" ]


검색된 문서를 제미나이에게 전달하기위해 contxt로 변환

In [46]:
const context = results
  .map((document) => document.pageContent)
  .join("\n\n---\n\n");

  console.log(context);


사야까은(는)
대구광역시 중구 공평동 8-6에 위치한
일식 음식점입니다.

사야까는 음식의 기본을 재료에서 찾는다.<br />일본처럼 300년, 400년 대를 이어 유지할 수 있는 음식점을 만들려면 가장 중요한 것은 당장의 음식 맛이 아니라 재료라고 생각한다.<br />그래서 이곳 음식점에는 사용하는 재료에 대한 안내서가 많이 눈에 띈다.

좌석 정보는 40석입니다.
주차 정보는 주변유료주차장입니다.
예약은 가능합니다.

지하철 1호선 중앙로역 2번 출구 약 160m
버스 정류장은 약령시앞 정류장이 가장 가깝습니다.

---

종로초밥은(는)
대구광역시 중구 종로1가 41-35에 위치한
일식 음식점입니다.

초밥과 함께 국물 맛이 산뜻하고 구수한 오뎅탕이 일품이다.편안한 분위기에서 부담없는 가격으로 즐길수 있다.

좌석 정보는 50석입니다.
주차 정보는 유료주차장입니다.
예약은 가능합니다.

지하철 1호선 중앙로역 1번 출구 약 260m
버스 정류장은 약령시건너(동성로입구)  정류장이 가장 가깝습니다.

---

디귿은(는)
대구광역시 중구 봉산동 35-34에 위치한
일식 음식점입니다.

디귿은 정통 일본식 돈카츠를 선보이는 맛집으로, 깔끔한 인테리어와 정성 어린 요리로 유명한 돈카츠 전문점입니다.

좌석 정보는 18석입니다.
주차 정보는 없음입니다.
예약은 가능합니다.

지하털 1,2호선 반월당역 10번 출구에서 도보로 약 88m 거리.
버스 정류장은 봉산문화거리건너 정류장이 가장 가깝습니다.


In [48]:
const question = "부모님 모시고 갈 조용한 식당을 추천해줘";

const response = await model.invoke(`
다음은 실제 음식점 데이터에서 검색한 정보입니다.

[검색 결과]

${context}


[사용자 질문]

${question}


검색 결과에 포함된 정보만 이용해서
적절한 식당을 추천해주세요.

검색 결과에 없는 정보는
추측하지 마세요.
`);

In [49]:
console.log(response.text);


부모님을 모시고 가기에 적합한 식당으로 **'사야까'**를 추천합니다.

추천 이유는 다음과 같습니다.

*   **음식 철학:** 사야까는 재료를 가장 중요하게 생각하며, 재료에 대한 안내서를 비치할 정도로 음식의 기본에 충실한 곳입니다. 부모님께 정성스럽고 건강한 음식을 대접하기에 적합합니다.
*   **편의성:** 예약이 가능하여 방문 전 미리 자리를 확보할 수 있습니다.
*   **접근성:** 지하철 1호선 중앙로역 2번 출구에서 약 160m 거리로 대중교통 이용이 편리합니다.

참고로, 사야까는 40석 규모이며 주차는 주변 유료주차장을 이용하셔야 합니다.


RAG의 흐름
사용자 질문

"부모님 모시고 갈 조용한 식당"

        ↓

Embedding

        ↓

Vector Store

        ↓

관련 식당 3곳

        ↓

Context

        ↓

Gemini

        ↓

최종 추천 답변